# 2. Four ways to bind a role

`banking/dci/` binds a role in four ways. They share the lifecycle and the `ContextVar` lookup, and differ only in how the role is attached:

- `Context` reassigns the object's class to a subclass synthesized from the role and the data class.
- `TransparentContext` does the same, and renames the synthesized class to the data class.
- `SerializedContext` does the same, and locks every participant for the duration of the use case.
- `InstanceContext` writes the role's methods into the instance's `__dict__`, and removes them on exit.

Each cell below measures one row of the comparison table in [APPROACHES.md](https://github.com/CesarBallardini/dci-in-python/blob/main/APPROACHES.md#the-four-compared). The rows about two live contexts at once are in notebook 3.

The first cell installs `banking` when the notebook runs in the browser. On the desktop it is already installed in the project environment, and the cell only reports the runtime.

In [ ]:
import sys

if sys.platform == 'emscripten':  # in the browser: this repository's wheel, served with the site
    import piplite

    await piplite.install(['banking'])

print('Python', sys.version.split()[0], 'on', sys.platform)

In [ ]:
import pickle
from typing import ClassVar

from banking.contexts.money_transfer import (
    InstanceMoneyTransferContext,
    MoneyTransferContext,
    SerializedMoneyTransferContext,
    TransparentMoneyTransferContext,
)
from banking.data.account import Account
from banking.data.money import Money
from banking.dci import Context, InstanceContext

BINDERS = {
    'Context': MoneyTransferContext,
    'TransparentContext': TransparentMoneyTransferContext,
    'SerializedContext': SerializedMoneyTransferContext,
    'InstanceContext': InstanceMoneyTransferContext,
}

## What the type reports, and what error messages name

Inside the context, the source account is asked for its type name and for an attribute it does not have.

In [ ]:
for name, binder in BINDERS.items():
    alice, bob = Account('Alice', Money(1000)), Account('Bob', Money(0))
    with binder(alice, bob, Money(10)):
        try:
            _ = alice.no_such_attribute
        except AttributeError as exc:
            message = str(exc)
        print(f'{name:<20} type: {type(alice).__name__:<22} error: {message}')

## Pickle while the role is bound

The class binders refuse at `dumps`, because the synthesized class exists in no module. `InstanceContext` accepts the `dumps` and writes bytes that fail at `loads`. [APPROACHES.md](https://github.com/CesarBallardini/dci-in-python/blob/main/APPROACHES.md#instancecontext--the-role-written-into-the-instance) explains why.

In [ ]:
for name, binder in BINDERS.items():
    alice, bob = Account('Alice', Money(1000)), Account('Bob', Money(0))
    with binder(alice, bob, Money(10)):
        try:
            blob = pickle.dumps(alice)
        except Exception as exc:
            print(f'{name:<20} dumps refuses: {type(exc).__name__}')
            continue
    try:
        pickle.loads(blob)
    except Exception as exc:
        print(f'{name:<20} dumps succeeds, loads fails: {type(exc).__name__}: {exc}')
    else:
        print(f'{name:<20} the round trip succeeds')

## A role that contributes a special method

Operators look special methods up on the type, never on the instance. A role defining `__lt__` therefore works where the role is mixed into a class, and cannot work where it is written into the instance. `InstanceContext` declines to bind it, rather than binding a method that would never run. The role below answers `True` to every comparison, so the result shows which `__lt__` ran.

In [ ]:
class ComparableAmount:
    def __lt__(self, other):
        return True


class ComparingContext(Context):
    roles: ClassVar[dict[str, type]] = {'amount': ComparableAmount}

    def __init__(self, amount):
        self.amount = amount


class ComparingInstanceContext(InstanceContext):
    roles: ClassVar[dict[str, type]] = {'amount': ComparableAmount}

    def __init__(self, amount):
        self.amount = amount


for name, binder in [('Context', ComparingContext), ('InstanceContext', ComparingInstanceContext)]:
    amount = Money(5)
    with binder(amount):
        print(f'{name:<20} inside the context, Money(5) < Money(1) -> {amount < Money(1)}')